# 01 — Discovery + Validation
Part of the Public Webcam Discovery System.

Runs DirectoryAgent (Tier 1 sources from SOURCES.md) to collect camera candidates, then ValidationAgent to confirm live feeds.


In [2]:
# ── Environment setup (run once) ─────────────────────────────
import subprocess, sys, os
from pathlib import Path

IN_COLAB     = "google.colab" in sys.modules
IN_SAGEMAKER = os.environ.get("SM_TRAINING_ENV") is not None

if IN_COLAB:
    if not Path("webcam-discovery").exists():
        subprocess.run(["git", "clone", "https://github.com/dshipley71/webcam-discovery"], check=True)
    os.chdir("webcam-discovery")

subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[notebooks]", "-q"], check=True)

from src.webcam_discovery.config import settings
settings.ensure_dirs()
print("✓ Ready")

✓ Ready


## Step 2 — Build environment

Install the package with dev dependencies so the pipeline and tests are available.

In [3]:
!pip install -e ".[dev]" -q

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for webcam-discovery (pyproject.toml) ... done


## Step 3 — Run DirectoryAgent (Tier 1)

Crawls all Tier 1 sources from `SOURCES.md`, resolves direct feed URLs via
`FeedExtractionSkill`, and writes results to `candidates/candidates.jsonl`.

Changes since last review
- Sources are now read from **SOURCES.md** at runtime (no hardcoded lists).
- **All 5 tiers** are supported; `--tier N` crawls tiers 1–N.
- `FeedExtractionSkill` now uses **finditer** (not early-return search), so listing pages with multiple embedded cameras surface all links.
- Shallow listing-page URLs (path depth < 3) that yielded no stream are **dropped**; only camera-specific pages and resolved stream URLs are kept.
- Each additional embedded link from a page becomes a **sub-candidate**, so a listing page with 20 camera iframes contributes 20 candidates.

In [4]:
!python scripts/run_discovery.py --tier 1 --output candidates/candidates.jsonl

2026-03-13 17:16:52.197 | INFO     | webcam_discovery.agents.directory_crawler:_parse:142 - SourcesRegistry: loaded tiers {1: 8, 2: 17, 3: 24, 4: 7, 5: 2} (58 total sources), 11 blocked domains from SOURCES.md
2026-03-13 17:16:52.198 | INFO     | webcam_discovery.agents.directory_crawler:run:222 - DirectoryAgent: tier=1 → 8 sources across 1 tier(s)
2026-03-13 17:16:52.653 | DEBUG    | webcam_discovery.agents.directory_crawler:run:245 - DirectoryAgent: robots.txt allows indy.com
2026-03-13 17:16:52.741 | WARNING  | webcam_discovery.skills.validation:run:268 - robots.txt fetch failed for ebcamtaxi.com: [Errno -2] Name or service not known
2026-03-13 17:16:52.742 | DEBUG    | webcam_discovery.agents.directory_crawler:run:245 - DirectoryAgent: robots.txt allows ebcamtaxi.com
2026-03-13 17:16:53.138 | DEBUG    | webcam_discovery.agents.directory_crawler:run:245 - DirectoryAgent: robots.txt allows skylinewebcams.com
2026-03-13 17:16:53.373 | DEBUG    | webcam_discovery.agents.directory_crawl

## Step 4 — Inspect candidates.jsonl

Preview candidate counts broken down by URL type:

| Type | Meaning |
|------|---------|
| `direct-stream` | HLS / MJPEG / MP4 — confirmed stream URL |
| `youtube-embed` | YouTube nocookie embed |
| `embed-page` | Third-party player embed page |
| `html-page` | Camera embed page (ValidationAgent will classify) |

In [5]:
import json, re
from pathlib import Path
from collections import Counter

output_path = Path("candidates/candidates.jsonl")

if not output_path.exists():
    print("candidates.jsonl not found — did the crawler run successfully?")
else:
    lines = output_path.read_text().splitlines()
    candidates = [json.loads(l) for l in lines if l.strip()]

    _stream_ext = re.compile(r'\.(m3u8|mjpg|mjpeg|mp4)(\?|$)', re.IGNORECASE)
    _youtube_re = re.compile(r'youtube(?:-nocookie)?\.com/embed/', re.IGNORECASE)
    _embed_re   = re.compile(r'/embed[/\?]|embed\.|\. embed', re.IGNORECASE)

    def url_type(url):
        if _stream_ext.search(url): return 'direct-stream'
        if _youtube_re.search(url): return 'youtube-embed'
        if _embed_re.search(url):   return 'embed-page'
        return 'html-page'

    types = Counter(url_type(c['url']) for c in candidates)

    print(f"Total candidates: {len(candidates)}")
    print(f"  direct-stream  : {types['direct-stream']:>4}  (HLS / MJPEG / MP4)")
    print(f"  youtube-embed  : {types['youtube-embed']:>4}  (YouTube nocookie embed)")
    print(f"  embed-page     : {types['embed-page']:>4}  (third-party player embed)")
    print(f"  html-page      : {types['html-page']:>4}  (camera embed page)")
    print()

    for c in candidates[:10]:
        t = url_type(c['url'])
        print(f"  [{t:<14}]  {c['url']}")
        print(f"               label={c.get('label','—')}  "
              f"city={c.get('city','—')}, {c.get('country','—')}")
        print()

Total candidates: 562
  direct-stream  :   12  (HLS / MJPEG / MP4)
  youtube-embed  :   39  (YouTube nocookie embed)
  embed-page     :    0  (third-party player embed)
  html-page      :  511  (camera embed page)

  [html-page     ]  https://www.windy.com
               label=www.windy.com  city=None, None

  [html-page     ]  https://www.skylinewebcams.com/
               label=None  city=None, None

  [html-page     ]  https://www.skylinewebcams.com
               label=Live Cams  city=None, None

  [html-page     ]  https://www.skylinewebcams.com/en/webcam/argentina.html
               label=Argentina  city=Argentina.Html, Webcam

  [html-page     ]  https://www.skylinewebcams.com/en/webcam/barbados.html
               label=Barbados  city=Barbados.Html, Webcam

  [html-page     ]  https://www.skylinewebcams.com/en/webcam/belize.html
               label=Belize  city=Belize.Html, Webcam

  [html-page     ]  https://www.skylinewebcams.com/en/webcam/bermuda.html
               label=

## Step 5 — Validate candidates

`ValidationAgent` checks each candidate URL with HTTP HEAD/GET, classifies the
feed type (HLS, MJPEG, YouTube, iframe, …), and geo-enriches with lat/lon.

| Score | Meaning |
|-------|---------|
| **high** | Direct media stream confirmed live |
| **medium** | HTML embed page — camera likely accessible |
| **low** | Auth-gated or dead — rejected |

For **html-page** candidates the agent fetches the page, detects the embedded
player type, and attempts geo lookup from the city/country extracted during
traversal. Results are written to `candidates/validated.jsonl`.

In [6]:
!python scripts/run_validation.py --input candidates/candidates.jsonl --output candidates/validated.jsonl

2026-03-13 17:21:17.287 | INFO     | webcam_discovery.agents.validator:run:85 - ValidationAgent: validating 562 candidates
2026-03-13 17:21:18.325 | WARNING  | webcam_discovery.skills.validation:run:268 - robots.txt fetch failed for : Request URL is missing an 'http://' or 'https://' protocol.
2026-03-13 17:21:20.629 | WARNING  | webcam_discovery.skills.validation:run:268 - robots.txt fetch failed for ebcam.nl: [Errno -2] Name or service not known
2026-03-13 17:21:20.715 | WARNING  | webcam_discovery.skills.validation:run:268 - robots.txt fetch failed for youtube-nocookie.com: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: Hostname mismatch, certificate is not valid for 'youtube-nocookie.com'. (_ssl.c:1010)
2026-03-13 17:21:20.862 | INFO     | webcam_discovery.agents.validator:run:116 - ValidationAgent: 562 candidates pass robots check (dropped 0)
2026-03-13 17:21:21.052 | WARNING  | webcam_discovery.skills.validation:_check:216 - Error validating files/videos/panomax/pano

## Step 6 — Inspect validated cameras

Pipeline funnel, feed-type breakdown, and first 10 confirmed camera records.

In [7]:
import json
from pathlib import Path
from collections import Counter

validated_path  = Path("candidates/validated.jsonl")
candidates_path = Path("candidates/candidates.jsonl")

if not validated_path.exists():
    print("validated.jsonl not found — did validation run successfully?")
else:
    records = [json.loads(l) for l in validated_path.read_text().splitlines() if l.strip()]

    n_cands = sum(1 for l in candidates_path.read_text().splitlines() if l.strip()) \
              if candidates_path.exists() else "?"

    print("── Pipeline funnel ──────────────────────────────")
    print(f"  Candidates (discovery) : {n_cands}")
    print(f"  Validated cameras      : {len(records)}")
    print()

    feed_types   = Counter(r['feed_type']        for r in records)
    legit_scores = Counter(r['legitimacy_score'] for r in records)
    statuses     = Counter(r['status']           for r in records)

    print("── By feed type ─────────────────────────────────")
    for ft, n in feed_types.most_common():
        print(f"  {ft:<16} : {n}")

    print()
    print("── By legitimacy score ──────────────────────────")
    for ls, n in legit_scores.most_common():
        print(f"  {ls:<8} : {n}")

    print()
    print("── By status ────────────────────────────────────")
    for st, n in statuses.most_common():
        print(f"  {st:<8} : {n}")

    print()
    print("── First 10 validated cameras ───────────────────")
    for r in records[:10]:
        print(f"  [{r['feed_type']:<14}] [{r['legitimacy_score']:<6}]  {r['url']}")
        print(f"    {r.get('label','—')} — {r.get('city','—')}, {r.get('country','—')}")
        print()

── Pipeline funnel ──────────────────────────────
  Candidates (discovery) : 562
  Validated cameras      : 4

── By feed type ─────────────────────────────────
  unknown          : 2
  iframe           : 2

── By legitimacy score ──────────────────────────
  medium   : 4

── By status ────────────────────────────────────
  unknown  : 2
  live     : 2

── First 10 validated cameras ───────────────────
  [unknown       ] [medium]  https://www.foto-webcam.eu/webcam/helmut-list-halle/
    Helmut List Halle — Helmut List Halle, Austria

  [unknown       ] [medium]  https://www.foto-webcam.eu/webcam/kueren/
    Kleinwalsertal — Kueren, Germany

  [iframe        ] [medium]  https://www.foto-webcam.eu/webcam/sarstein/
    Sarstein — Sarstein, Austria

  [iframe        ] [medium]  https://www.foto-webcam.eu/webcam/mayrhofen/
    Mayrhofen — Mayrhofen, Austria

